# 2D Kolmogorov Navier–Stokes and dissipative SQG: dataset generation

Data-only extraction from `GRNO_Kolmogorov_NS_and_SQG_128(1).ipynb`. Numerical solver implementations, physical parameters, split seeds, and original profile defaults are preserved. Neural models, training, checkpoint evaluation, and model-result plotting are omitted. Source cell numbers are recorded in each code cell's metadata.

The saved default is `PROFILE = "pilot"`. Read the profile table in [README.md](README.md) before running: the `paper` profile can require substantial GPU time and disk space. Install a compatible PyTorch build plus NumPy, pandas, tqdm, and IPython in this notebook's kernel. These notebooks do not install packages automatically.

Run from the repository, or set the environment variable `GRNO_ROOT` to your intended project/data location before starting the kernel. The printed `ROOT` and data roots identify the generated files. Restart and run from the top after changing profiles. Set `RUN_DATA_GENERATION = False` to inspect a complete existing cache; simulator audits and normalization still run.

In this source, `train_frames`, `val_frames`, and `test_frames` count **transitions**, so each saved trajectory contains one additional initial frame. Keep the paper values 50/100/100 to obtain 51/101/101 stored frames. `RunConfig` retains legacy model/training fields because it is written into the original manifest; they are inactive here.


In [ ]:
from __future__ import annotations

import contextlib
import dataclasses
import hashlib
import json
import math
import os
import random
import shutil
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from IPython.display import display
from tqdm.auto import tqdm

# Storage location only: set GRNO_ROOT before starting this kernel to override it.
# No solver, split, seed, or physical parameter is changed by this root selection.
def locate_project_root() -> Path:
    override = os.environ.get("GRNO_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "README.md").is_file() and (candidate / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate
    return cwd

ROOT = locate_project_root()
ROOT.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print({"project_root": str(ROOT), "device": str(device), "torch": torch.__version__})


## Original profiles and physical parameters


In [ ]:
PROFILE = "pilot"       # "smoke", "pilot", or "paper"


DATASET_VERSION = "active_scalar_etdrk2_v3"


RUN_DATA_GENERATION = True


@dataclass(frozen=True)
class PDEConfig:
    name: str
    state_symbol: str
    green_kind: str
    n: int = 128
    domain_length: float = 2.0 * math.pi
    frame_dt: float = 0.1
    substeps: int = 20
    diffusivity: float = 1.0e-2
    dissipation_power: float = 1.0
    linear_drag: float = -0.1
    hypofriction: float = 0.0
    forcing_kind: str = "kolmogorov"
    forcing_rms: float = 1.0
    forcing_mode: int = 4
    dealias_fraction: float = 2.0 / 3.0
    post_dealias: bool = True
    ic_cutoff: int = 8
    ic_rms: float = 1.0
    warmup_frames: int = 100


@dataclass(frozen=True)
class RunConfig:
    profile: str
    n_train: int
    n_val: int
    n_test: int
    train_frames: int
    val_frames: int
    test_frames: int
    epochs: int
    samples_per_epoch: int
    batch_size: int
    train_rollout: int
    val_rollout: int
    eval_rollout: int
    width: int
    depths: tuple
    learning_rate: float
    weight_decay: float
    grad_weight: float
    spectral_weight: float
    num_workers: int = 0
    use_amp: bool = True
    output_dir: str = "grno_ns_sqg_outputs"
    data_dir: str = "grno_ns_sqg_data"


if PROFILE == "smoke":
    run_cfg = RunConfig(
        PROFILE, n_train=2, n_val=1, n_test=1,
        train_frames=6, val_frames=8, test_frames=10,
        epochs=2, samples_per_epoch=24, batch_size=2,
        train_rollout=1, val_rollout=2, eval_rollout=5,
        width=16, depths=(1, 1, 1, 2), learning_rate=1e-3,
        weight_decay=1e-4, grad_weight=0.03, spectral_weight=0.0,
        use_amp=False,
    )
    ns_cfg = PDEConfig(
        "kolmogorov_ns", "omega", "poisson", warmup_frames=2,
        # Debug with less data/training, not a less trustworthy simulator.
        substeps=20, post_dealias=True,
    )
    sqg_cfg = PDEConfig(
        "dissipative_sqg", "theta", "sqg", frame_dt=0.1,
        # Keep the physically meaningful internal SQG time step even in the
        # debug run; reducing the dataset is safe, reducing these substeps is not.
        substeps=40, diffusivity=8.0e-3, dissipation_power=1.0,
        linear_drag=0.0, hypofriction=0.15,
        forcing_kind="sqg_shell", forcing_rms=0.10,
        ic_rms=0.5, warmup_frames=2,
    )
elif PROFILE == "pilot":
    run_cfg = RunConfig(
        PROFILE, n_train=20, n_val=4, n_test=6,
        train_frames=36, val_frames=60, test_frames=60,
        epochs=40, samples_per_epoch=512, batch_size=4,
        train_rollout=2, val_rollout=5, eval_rollout=20,
        width=32, depths=(2, 2, 2, 3), learning_rate=4e-4,
        weight_decay=1e-4, grad_weight=0.05, spectral_weight=0.02,
    )
    ns_cfg = PDEConfig(
        "kolmogorov_ns", "omega", "poisson", warmup_frames=100,
        post_dealias=True,
    )
    sqg_cfg = PDEConfig(
        "dissipative_sqg", "theta", "sqg", frame_dt=0.1,
        substeps=40, diffusivity=8.0e-3, dissipation_power=1.0,
        linear_drag=0.0, hypofriction=0.15,
        forcing_kind="sqg_shell", forcing_rms=0.10,
        ic_rms=0.5, warmup_frames=200,
    )
elif PROFILE == "paper":
    run_cfg = RunConfig(
        PROFILE, n_train=64, n_val=8, n_test=12,
        train_frames=50, val_frames=100, test_frames=100,
        epochs=150, samples_per_epoch=2048, batch_size=4,
        train_rollout=4, val_rollout=10, eval_rollout=40,
        width=48, depths=(2, 2, 3, 4), learning_rate=3e-4,
        weight_decay=1e-4, grad_weight=0.05, spectral_weight=0.02,
    )
    ns_cfg = PDEConfig(
        "kolmogorov_ns", "omega", "poisson", warmup_frames=500,
        post_dealias=True,
    )
    sqg_cfg = PDEConfig(
        "dissipative_sqg", "theta", "sqg", frame_dt=0.1,
        substeps=40, diffusivity=8.0e-3, dissipation_power=1.0,
        linear_drag=0.0, hypofriction=0.15,
        forcing_kind="sqg_shell", forcing_rms=0.10,
        ic_rms=0.5, warmup_frames=200,
    )
else:
    raise ValueError(PROFILE)


PDE_CONFIGS = {c.name: c for c in (ns_cfg, sqg_cfg)}


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


if PROFILE != "smoke" and device.type != "cuda":
    raise RuntimeError("Pilot/paper profiles require a CUDA GPU. Set PROFILE='smoke' for CPU debugging.")


torch.backends.cuda.matmul.allow_tf32 = True


torch.backends.cudnn.allow_tf32 = True


assert min(c.n for c in PDE_CONFIGS.values()) >= 128


# Only relocate the legacy relative data_dir; dataset fingerprints exclude this path.
run_cfg = dataclasses.replace(run_cfg, data_dir=str(ROOT / run_cfg.data_dir))
print(json.dumps(asdict(run_cfg), indent=2))
for name, cfg in PDE_CONFIGS.items():
    print(name, json.dumps(asdict(cfg), indent=2))


## Utilities and reference solver


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def short_hash(payload):
    raw = json.dumps(payload, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:12]


def spectral_grid(n, L, device, dtype=torch.float32):
    # Tensor layout: [..., y, x]; rFFT is performed along x.
    ky_1d = 2.0 * math.pi * torch.fft.fftfreq(n, d=L / n, device=device, dtype=dtype)
    kx_1d = 2.0 * math.pi * torch.fft.rfftfreq(n, d=L / n, device=device, dtype=dtype)
    ky, kx = torch.meshgrid(ky_1d, kx_1d, indexing="ij")
    return ky, kx


def relative_l2(pred, target, eps=1e-12):
    num = (pred - target).square().flatten(1).sum(1)
    den = target.square().flatten(1).sum(1).clamp_min(eps)
    return torch.sqrt(num / den)


In [ ]:
class ActiveScalarETDRK2:
    """Batched periodic pseudo-spectral solver for NS-vorticity or SQG."""

    def __init__(self, cfg, device=device, dtype=torch.float32):
        self.cfg = cfg
        self.device = torch.device(device)
        self.dtype = dtype
        self.cdtype = torch.complex64 if dtype == torch.float32 else torch.complex128
        self.h = cfg.frame_dt / cfg.substeps
        n, L = cfg.n, cfg.domain_length
        self.ky, self.kx = spectral_grid(n, L, self.device, dtype)
        self.k2 = self.kx.square() + self.ky.square()
        self.kabs = torch.sqrt(self.k2)

        cutoff = cfg.dealias_fraction * (n // 2)
        ky_index = torch.fft.fftfreq(n, d=1.0 / n, device=self.device, dtype=dtype)
        kx_index = torch.fft.rfftfreq(n, d=1.0 / n, device=self.device, dtype=dtype)
        self.dealias_mask = ((ky_index[:, None].abs() <= cutoff) &
                             (kx_index[None, :].abs() <= cutoff))
        self.ic_mask = ((ky_index[:, None].abs() <= cfg.ic_cutoff) &
                        (kx_index[None, :].abs() <= cfg.ic_cutoff))
        self.sqg_force_shell = ((torch.sqrt(
            ky_index[:, None].square() + kx_index[None, :].square()) >= 3.0) &
            (torch.sqrt(
                ky_index[:, None].square() + kx_index[None, :].square()) <= 4.0))

        dissipation = cfg.diffusivity * self.kabs.pow(2.0 * cfg.dissipation_power)
        hypofriction = torch.zeros_like(self.k2)
        nz = self.k2 > 0
        if cfg.hypofriction:
            hypofriction[nz] = cfg.hypofriction / self.k2[nz]
        self.linear = cfg.linear_drag - dissipation - hypofriction

        self.psi_multiplier = torch.zeros_like(self.k2)
        if cfg.green_kind == "poisson":
            # Delta psi = omega -> psi_hat = -omega_hat / |k|^2.
            self.psi_multiplier[nz] = -1.0 / self.k2[nz]
        elif cfg.green_kind == "sqg":
            # psi = -Lambda^{-1} theta under the shared velocity convention.
            self.psi_multiplier[nz] = -1.0 / self.kabs[nz]
        else:
            raise ValueError(cfg.green_kind)

        self._build_etdrk2_coefficients()

    def _build_etdrk2_coefficients(self, contour_points=16, radius=1.0):
        j = torch.arange(1, contour_points + 1, device=self.device, dtype=self.dtype)
        roots = torch.exp(1j * 2.0 * math.pi * (j - 0.5) / contour_points).to(self.cdtype)
        z = (self.h * self.linear).to(self.cdtype)
        lr = z[None] + radius * roots[:, None, None]
        exp_lr = torch.exp(lr)
        phi1 = ((exp_lr - 1.0) / lr).real.mean(dim=0)
        phi2 = ((exp_lr - 1.0 - lr) / lr.square()).real.mean(dim=0)
        self.exp_linear = torch.exp(self.h * self.linear)
        self.coef1 = self.h * phi1
        self.coef2 = self.h * phi2

    def carrier_from_hat(self, q_hat):
        psi_hat = self.psi_multiplier * q_hat
        # u is x velocity; v is y velocity. Convention: (psi_y, -psi_x).
        u = torch.fft.irfft2(1j * self.ky * psi_hat, s=(self.cfg.n, self.cfg.n))
        v = torch.fft.irfft2(-1j * self.kx * psi_hat, s=(self.cfg.n, self.cfg.n))
        return torch.stack((u, v), dim=1)

    def carrier(self, q):
        return self.carrier_from_hat(torch.fft.rfft2(q))

    @torch.no_grad()
    def random_initial_hat(self, batch_size, seed):
        generator = torch.Generator(device=self.device)
        generator.manual_seed(int(seed))
        raw = torch.randn(batch_size, self.cfg.n, self.cfg.n,
                          device=self.device, dtype=self.dtype, generator=generator)
        q_hat = torch.fft.rfft2(raw) * self.ic_mask
        q_hat[..., 0, 0] = 0.0
        q = torch.fft.irfft2(q_hat, s=(self.cfg.n, self.cfg.n))
        q = q - q.mean((-2, -1), keepdim=True)
        q = q * (self.cfg.ic_rms /
                 q.square().mean((-2, -1), keepdim=True).sqrt().clamp_min(1e-8))
        return torch.fft.rfft2(q)

    @torch.no_grad()
    def forcing_batch(self, batch_size, seed):
        c, n, L = self.cfg, self.cfg.n, self.cfg.domain_length
        if c.forcing_kind == "kolmogorov":
            y = torch.arange(n, device=self.device, dtype=self.dtype) * L / n
            force = -c.forcing_mode * torch.cos(
                c.forcing_mode * 2.0 * math.pi * y / L)
            force = force[:, None].expand(n, n)
            return force[None].expand(batch_size, -1, -1).contiguous()
        if c.forcing_kind == "sqg_shell":
            generator = torch.Generator(device=self.device)
            generator.manual_seed(int(seed))
            raw = torch.randn(batch_size, n, n, device=self.device,
                              dtype=self.dtype, generator=generator)
            force_hat = torch.fft.rfft2(raw) * self.sqg_force_shell
            force_hat[..., 0, 0] = 0.0
            force = torch.fft.irfft2(force_hat, s=(n, n))
            force = force - force.mean((-2, -1), keepdim=True)
            force = force * (c.forcing_rms /
                             force.square().mean((-2, -1), keepdim=True)
                             .sqrt().clamp_min(1e-8))
            return force
        raise ValueError(c.forcing_kind)

    @torch.no_grad()
    def nonlinear(self, q_hat, forcing_hat):
        filtered = q_hat * self.dealias_mask
        uv = self.carrier_from_hat(filtered)
        qx = torch.fft.irfft2(1j * self.kx * filtered,
                              s=(self.cfg.n, self.cfg.n))
        qy = torch.fft.irfft2(1j * self.ky * filtered,
                              s=(self.cfg.n, self.cfg.n))
        adv_hat = torch.fft.rfft2(uv[:, 0] * qx + uv[:, 1] * qy)
        if self.cfg.post_dealias:
            adv_hat = adv_hat * self.dealias_mask
        return -adv_hat + forcing_hat

    @torch.no_grad()
    def substep(self, q_hat, forcing_hat):
        n0 = self.nonlinear(q_hat, forcing_hat)
        stage = self.exp_linear * q_hat + self.coef1 * n0
        n1 = self.nonlinear(stage, forcing_hat)
        out = stage + self.coef2 * (n1 - n0)
        out[..., 0, 0] = 0.0
        return out

    @torch.no_grad()
    def advance_frame(self, q_hat, forcing_hat):
        for _ in range(self.cfg.substeps):
            q_hat = self.substep(q_hat, forcing_hat)
        return q_hat

    @torch.no_grad()
    def internal_cfl(self, q_hat):
        uv = self.carrier_from_hat(q_hat)
        speed = uv.square().sum(1).sqrt()
        return speed * self.h / (self.cfg.domain_length / self.cfg.n)

    @torch.no_grad()
    def assert_resolved_timestep(self, q_hat, where):
        # SQG has a more singular Green multiplier than 2D NS.  A trajectory
        # that grows past this guard is rejected instead of silently becoming
        # under-resolved supervision for the neural operator.
        if self.cfg.name != "dissipative_sqg":
            return
        cfl = self.internal_cfl(q_hat)
        cfl_max = float(cfl.max())
        cfl_p99 = float(torch.quantile(cfl, 0.99))
        if cfl_p99 >= 0.35 or cfl_max >= 0.75:
            raise RuntimeError(
                f"SQG internal CFL too high at {where}: "
                f"p99={cfl_p99:.3f}, max={cfl_max:.3f}. "
                "Increase substeps before using this trajectory.")

    @torch.no_grad()
    def generate_split(self, state_path, forcing_path, n_trajectories,
                       frames, seed, chunk_size=8):
        state_path, forcing_path = Path(state_path), Path(forcing_path)
        state_path.parent.mkdir(parents=True, exist_ok=True)
        # Write to temporary memmaps.  An interrupted simulation must never be
        # mistaken for a valid cached dataset on the next notebook run.
        state_tmp = state_path.with_name(state_path.stem + ".partial.npy")
        force_tmp = forcing_path.with_name(forcing_path.stem + ".partial.npy")
        for partial in (state_tmp, force_tmp):
            if partial.exists():
                partial.unlink()
        state_shape = (n_trajectories, frames + 1, self.cfg.n, self.cfg.n)
        force_shape = (n_trajectories, self.cfg.n, self.cfg.n)
        states = np.lib.format.open_memmap(
            state_tmp, mode="w+", dtype=np.float32, shape=state_shape)
        forces = np.lib.format.open_memmap(
            force_tmp, mode="w+", dtype=np.float32, shape=force_shape)

        for start in tqdm(range(0, n_trajectories, chunk_size),
                          desc=f"{self.cfg.name}:{state_path.stem}"):
            count = min(chunk_size, n_trajectories - start)
            local_seed = seed + 1009 * start
            q_hat = self.random_initial_hat(count, local_seed)
            force = self.forcing_batch(count, local_seed + 53)
            forcing_hat = torch.fft.rfft2(force)
            for _ in range(self.cfg.warmup_frames):
                q_hat = self.advance_frame(q_hat, forcing_hat)
            self.assert_resolved_timestep(q_hat, f"chunk {start} after warmup")
            states[start:start + count, 0] = torch.fft.irfft2(
                q_hat, s=(self.cfg.n, self.cfg.n)).cpu().numpy()
            forces[start:start + count] = force.cpu().numpy()
            for t in range(1, frames + 1):
                q_hat = self.advance_frame(q_hat, forcing_hat)
                if not torch.isfinite(q_hat).all():
                    raise FloatingPointError(
                        f"Non-finite {self.cfg.name} state, chunk {start}, frame {t}")
                self.assert_resolved_timestep(q_hat, f"chunk {start}, frame {t}")
                states[start:start + count, t] = torch.fft.irfft2(
                    q_hat, s=(self.cfg.n, self.cfg.n)).cpu().numpy()
        states.flush()
        forces.flush()
        del states, forces
        state_tmp.replace(state_path)
        force_tmp.replace(forcing_path)
        print("saved", state_path, state_shape, "and", forcing_path, force_shape)


## Simulator audits


In [ ]:
@torch.no_grad()
def validate_simulator(solver):
    c, n = solver.cfg, solver.cfg.n
    q_hat = solver.random_initial_hat(3, 2027)
    force = solver.forcing_batch(3, 2080)
    force_hat = torch.fft.rfft2(force)
    q = torch.fft.irfft2(q_hat, s=(n, n))
    uv = solver.carrier_from_hat(q_hat)
    uv_hat = torch.fft.rfft2(uv)
    divergence = torch.fft.irfft2(
        1j * solver.kx * uv_hat[:, 0] + 1j * solver.ky * uv_hat[:, 1],
        s=(n, n),
    )
    div_rel = (divergence.square().mean().sqrt() /
               uv.square().mean().sqrt().clamp_min(1e-8)).item()

    q_next_hat = solver.advance_frame(q_hat, force_hat)
    q_next = torch.fft.irfft2(q_next_hat, s=(n, n))
    fine_cfg = PDEConfig(**{
        **asdict(c), "substeps": 2 * c.substeps,
    })
    fine_solver = ActiveScalarETDRK2(
        fine_cfg, device=solver.device, dtype=solver.dtype)
    q_fine_hat = fine_solver.advance_frame(q_hat.clone(), force_hat)
    q_fine = torch.fft.irfft2(q_fine_hat, s=(n, n))
    refinement_error = float(relative_l2(
        q_next[:, None], q_fine[:, None]).mean())
    mean_error = q_next.mean((-2, -1)).abs().max().item()
    finite = bool(torch.isfinite(q_next).all())
    rms_ratio = float(q_next.square().mean().sqrt() /
                      q.square().mean().sqrt().clamp_min(1e-8))
    cfl_internal = solver.internal_cfl(q_hat)
    result = {
        "pde": c.name,
        "grid": n,
        "relative_divergence": div_rel,
        "one_frame_mean_error": mean_error,
        "one_frame_rms_ratio": rms_ratio,
        "temporal_refinement_relative_l2": refinement_error,
        "internal_CFL_mean": float(cfl_internal.mean()),
        "internal_CFL_p99": float(torch.quantile(cfl_internal, 0.99)),
        "internal_CFL_max": float(cfl_internal.max()),
        "finite": finite,
    }
    print(json.dumps(result, indent=2))
    assert n >= 128 and finite and div_rel < 1e-4 and mean_error < 1e-5
    assert 0.25 < rms_ratio < 4.0
    assert refinement_error < 5e-3, (
        "One-frame temporal refinement error is too high; increase substeps.")
    if c.name == "dissipative_sqg":
        assert (result["internal_CFL_p99"] < 0.35 and
                result["internal_CFL_max"] < 0.75), (
            "SQG internal CFL is too high; increase substeps before generating data.")
    return result


solvers = {name: ActiveScalarETDRK2(cfg, device=device)
           for name, cfg in PDE_CONFIGS.items()}
simulator_checks = {name: validate_simulator(solver)
                    for name, solver in solvers.items()}


## Generate/cache split arrays and manifests


In [ ]:
def dataset_paths_for(cfg, run_cfg):
    fingerprint = short_hash({
        "dataset_version": DATASET_VERSION,
        "pde": asdict(cfg),
        "split_sizes": {
            "train": [run_cfg.n_train, run_cfg.train_frames],
            "val": [run_cfg.n_val, run_cfg.val_frames],
            "test": [run_cfg.n_test, run_cfg.test_frames],
        },
    })
    root = Path(run_cfg.data_dir) / f"{cfg.name}_{run_cfg.profile}_{fingerprint}"
    root.mkdir(parents=True, exist_ok=True)
    return {
        "root": root,
        "train": root / "train_states.npy",
        "train_forcing": root / "train_forcing.npy",
        "val": root / "val_states.npy",
        "val_forcing": root / "val_forcing.npy",
        "test": root / "test_states.npy",
        "test_forcing": root / "test_forcing.npy",
        "manifest": root / "manifest.json",
    }


DATA_PATHS = {name: dataset_paths_for(cfg, run_cfg)
              for name, cfg in PDE_CONFIGS.items()}

split_specs = {
    "train": (run_cfg.n_train, run_cfg.train_frames, 1103),
    "val": (run_cfg.n_val, run_cfg.val_frames, 2207),
    "test": (run_cfg.n_test, run_cfg.test_frames, 3301),
}

if RUN_DATA_GENERATION:
    for name, solver in solvers.items():
        paths = DATA_PATHS[name]
        for split, (n_traj, frames, seed) in split_specs.items():
            force_key = f"{split}_forcing"
            if not paths[split].exists() or not paths[force_key].exists():
                solver.generate_split(
                    paths[split], paths[force_key], n_traj, frames, seed)
            else:
                print("using cached", paths[split], paths[force_key])
        manifest = {
            "dataset_version": DATASET_VERSION,
            "pde": asdict(PDE_CONFIGS[name]),
            "run": asdict(run_cfg),
            "files": {k: str(v) for k, v in paths.items() if k != "root"},
            "state_layout": "[trajectory, time, y, x]",
            "forcing_layout": "[trajectory, y, x]",
            "dtype": "float32",
        }
        paths["manifest"].write_text(json.dumps(manifest, indent=2))

for name, paths in DATA_PATHS.items():
    for split in ("train", "val", "test"):
        force_key = f"{split}_forcing"
        if not paths[split].exists() or not paths[force_key].exists():
            raise FileNotFoundError(
                f"Missing {split} data for {name}. Enable RUN_DATA_GENERATION.")
        arr = np.load(paths[split], mmap_mode="r")
        force = np.load(paths[force_key], mmap_mode="r")
        print(name, split, arr.shape, force.shape,
              "GB=", round((arr.nbytes + force.nbytes) / 1024**3, 3),
              "finite=", bool(np.isfinite(arr[0]).all()))


## Training-only normalization required by the released reader


In [ ]:
@dataclass(frozen=True)
class NormStats:
    state_std: float
    velocity_std: float
    forcing_std: float
    delta_std: float


@torch.no_grad()
def estimate_norm_stats(pde_name, max_frames=128, chunk=16):
    states = np.load(DATA_PATHS[pde_name]["train"], mmap_mode="r")
    forces = np.load(DATA_PATHS[pde_name]["train_forcing"], mmap_mode="r")
    solver = solvers[pde_name]

    selected = []
    for traj in range(states.shape[0]):
        for t in range(states.shape[1] - 1):
            selected.append((traj, t))
    if len(selected) > max_frames:
        # Deterministic stratification across trajectories and time. A prefix
        # would estimate normalization from only the first few trajectories.
        chosen = np.linspace(
            0, len(selected) - 1, int(max_frames), dtype=np.int64)
        selected = [selected[int(i)] for i in chosen]

    state_sum2 = velocity_sum2 = force_sum2 = delta_sum2 = 0.0
    state_count = velocity_count = force_count = delta_count = 0
    for start in range(0, len(selected), chunk):
        ids = selected[start:start + chunk]
        q = torch.from_numpy(
            np.stack([np.asarray(states[i, t]) for i, t in ids])).to(device)
        target = torch.from_numpy(
            np.stack([np.asarray(states[i, t + 1]) for i, t in ids])).to(device)
        f = torch.from_numpy(
            np.stack([np.asarray(forces[i]) for i, _ in ids])).to(device)
        uv = solver.carrier(q)
        state_sum2 += float(q.double().square().sum())
        state_count += q.numel()
        velocity_sum2 += float(uv.double().square().sum())
        velocity_count += uv.numel()
        force_sum2 += float(f.double().square().sum())
        force_count += f.numel()
        delta = target - q
        delta_sum2 += float(delta.double().square().sum())
        delta_count += delta.numel()

    stats = NormStats(
        state_std=max(math.sqrt(state_sum2 / state_count), 1e-6),
        velocity_std=max(math.sqrt(velocity_sum2 / velocity_count), 1e-6),
        forcing_std=max(math.sqrt(force_sum2 / force_count), 1e-6),
        delta_std=max(math.sqrt(delta_sum2 / delta_count), 1e-6),
    )
    path = DATA_PATHS[pde_name]["root"] / "normalization.json"
    path.write_text(json.dumps(asdict(stats), indent=2))
    print(pde_name, asdict(stats))
    return stats


NORM_STATS = {name: estimate_norm_stats(name) for name in PDE_CONFIGS}


## Saved-dynamics diagnostics


In [ ]:
@torch.no_grad()
def radial_tail_fraction(q, retained_fraction=0.8):
    """Energy near, and outside, the simulator's rectangular 2/3 mask."""
    q_hat = torch.fft.fft2(q)
    n = q.shape[-1]
    ky = torch.fft.fftfreq(n, d=1.0 / n, device=q.device)
    kx = torch.fft.fftfreq(n, d=1.0 / n, device=q.device)
    ky, kx = torch.meshgrid(ky, kx, indexing="ij")
    k_axis = torch.maximum(kx.abs(), ky.abs())
    retained_max = (2.0 / 3.0) * (n // 2)
    tail = ((k_axis >= retained_fraction * retained_max) &
            (k_axis <= retained_max))
    resolved = k_axis <= retained_max
    energy = q_hat.abs().square()
    resolved_energy = energy[..., resolved].sum(-1).clamp_min(1e-12)
    total_energy = energy.sum((-2, -1)).clamp_min(1e-12)
    upper_fraction = energy[..., tail].sum(-1) / resolved_energy
    outside_fraction = energy[..., ~resolved].sum(-1) / total_energy
    return upper_fraction, outside_fraction


@torch.no_grad()
def dataset_diagnostics(pde_name, split="train", max_pairs=64):
    cfg = PDE_CONFIGS[pde_name]
    states = np.load(DATA_PATHS[pde_name][split], mmap_mode="r")
    forces = np.load(DATA_PATHS[pde_name][f"{split}_forcing"], mmap_mode="r")
    pairs = []
    for i in range(states.shape[0]):
        for t in range(states.shape[1] - 1):
            pairs.append((i, t))
    if len(pairs) > max_pairs:
        chosen = np.linspace(
            0, len(pairs) - 1, int(max_pairs), dtype=np.int64)
        pairs = [pairs[int(i)] for i in chosen]
    q = torch.from_numpy(np.stack(
        [np.asarray(states[i, t]) for i, t in pairs])).to(device)
    target = torch.from_numpy(np.stack(
        [np.asarray(states[i, t + 1]) for i, t in pairs])).to(device)
    uv = solvers[pde_name].carrier(q)
    output_cfl = uv.square().sum(1).sqrt() * cfg.frame_dt / (
        cfg.domain_length / cfg.n)
    persistence = relative_l2(q[:, None], target[:, None])
    tail, outside = radial_tail_fraction(target)

    # Re-test the actual warmed/stored regime with half the internal time step.
    probe_positions = np.linspace(
        0, len(pairs) - 1, min(3, len(pairs)), dtype=np.int64)
    q_probe = q[torch.as_tensor(probe_positions, device=device)]
    f_probe = torch.from_numpy(np.stack([
        np.asarray(forces[pairs[int(j)][0]]) for j in probe_positions
    ])).to(device)
    q_probe_hat = torch.fft.rfft2(q_probe)
    f_probe_hat = torch.fft.rfft2(f_probe)
    coarse_hat = solvers[pde_name].advance_frame(
        q_probe_hat.clone(), f_probe_hat)
    fine_cfg = PDEConfig(**{
        **asdict(cfg), "substeps": 2 * cfg.substeps,
    })
    fine_solver = ActiveScalarETDRK2(fine_cfg, device=device)
    fine_hat = fine_solver.advance_frame(q_probe_hat.clone(), f_probe_hat)
    coarse = torch.fft.irfft2(coarse_hat, s=(cfg.n, cfg.n))
    fine = torch.fft.irfft2(fine_hat, s=(cfg.n, cfg.n))
    refinement = relative_l2(coarse[:, None], fine[:, None])
    result = {
        "pde": pde_name,
        "split": split,
        "persistence_rel_l2_mean": float(persistence.mean()),
        "persistence_rel_l2_median": float(persistence.median()),
        "output_displacement_cells_mean": float(output_cfl.mean()),
        "output_displacement_cells_p50": float(torch.quantile(output_cfl, 0.50)),
        "output_displacement_cells_p95": float(torch.quantile(output_cfl, 0.95)),
        "upper_resolved_band_energy_fraction_mean": float(tail.mean()),
        "outside_dealias_mask_energy_fraction_mean": float(outside.mean()),
        "post_warmup_temporal_refinement_l2_mean": float(refinement.mean()),
        "post_warmup_temporal_refinement_l2_max": float(refinement.max()),
    }
    print(json.dumps(result, indent=2))
    if PROFILE != "smoke":
        if result["persistence_rel_l2_mean"] < 0.03:
            print("WARNING: saved interval may be too close to persistence.")
        if result["upper_resolved_band_energy_fraction_mean"] > 1e-3:
            print("WARNING: inspect the spectrum for a high-k pile-up.")
        if result["outside_dealias_mask_energy_fraction_mean"] > 1e-8:
            raise RuntimeError("Stored states contain energy outside the dealias mask.")
        if result["post_warmup_temporal_refinement_l2_max"] > 5e-3:
            raise RuntimeError(
                "Warmed-state dt-vs-dt/2 error exceeds 0.5%; increase substeps.")
    return result


DATA_DIAGNOSTICS = {
    name: dataset_diagnostics(name, "train") for name in PDE_CONFIGS
}
